## Ejercicios Prácticos con RDDs

1. **Creación de RDD básico:** Cree un RDD llamado `lenguajes` que contenga los siguientes lenguajes de programación: Python, R, C, Scala, Ruby y SQL. *(Nota: Se corrigió "Rugby" por "Ruby")*.

2. **Transformación (Mayúsculas):** Obtenga un nuevo RDD a partir del RDD `lenguajes` donde todos los lenguajes de programación estén en mayúsculas.

3. **Transformación (Minúsculas):** Obtenga un nuevo RDD a partir del RDD `lenguajes` donde todos los lenguajes de programación estén en minúsculas.

4. **Filtrado:** Cree un nuevo RDD que solo contenga aquellos lenguajes de programación que comiencen con la letra "R".

5. **Generación de datos numéricos:** Cree un RDD llamado `pares` que contenga los números pares existentes en el intervalo [20, 30].

6. **Operaciones matemáticas:** Cree un RDD llamado `sqrt`; este debe contener la raíz cuadrada de los elementos que componen el RDD `pares`.

7. **Manipulación y formateo:** Obtenga una lista compuesta por los números pares en el intervalo [20, 30] y sus respectivas raíces cuadradas.

   * *Ejemplo del resultado esperado para el intervalo [50, 60]:* `[50, 7.0710678118654755, 52, 7.211102550927978, 54, 7.3484692283495345, 56, 7.483314773547883, 58, 7.615773105863909, 60, 7.745966692414834]`
8. **Redistribución (Aumento de particiones):** Eleve el número de particiones del RDD `sqrt` a 20.

9. **Optimización de particiones:** Si tuviera que disminuir el número de particiones luego de haberlo establecido en 20, ¿qué función utilizaría para hacer más eficiente su código?

10. **RDD de tipo Clave-Valor (Pair RDD):** Cree un RDD de tipo clave-valor a partir de los datos adjuntos como recurso a esta lección. Tenga en cuenta que deberá procesar el RDD leído para obtener el resultado solicitado. Supongamos que el RDD resultante refleja las transacciones realizadas por número de cuenta; obtenga el monto total por cada cuenta.

> 💡 **Tip:** Cree su propia función personalizada para procesar y limpiar el RDD leído antes de realizar la agregación.

In [6]:
!pip install findspark
import findspark
findspark.init()

In [8]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Ejercicio Parte 2 ").getOrCreate()

In [10]:
sc = spark.sparkContext

In [13]:
lenguajes = sc.parallelize(["Python","R","Scala","C","Rugby","SQL"])

In [14]:
lenguajes.collect()

['Python', 'R', 'Scala', 'C', 'Rugby', 'SQL']

In [16]:
lenguaje_mayus = lenguajes.map(lambda x: x.upper())

lenguaje_mayus.collect()

['PYTHON', 'R', 'SCALA', 'C', 'RUGBY', 'SQL']

In [18]:
lenguajes_minus = lenguajes.map(lambda x: x.lower())

lenguajes_minus.collect()

['python', 'r', 'scala', 'c', 'rugby', 'sql']

In [19]:
rdd_letra_r = lenguajes.filter(lambda x: x.startswith("R"))

rdd_letra_r.collect()

['R', 'Rugby']

In [21]:
pares = sc.parallelize([20,22,24,26,28,30])

pares.collect()

[20, 22, 24, 26, 28, 30]

In [24]:
import math

sqrt = pares.map(lambda x: math.sqrt(x))

sqrt.collect()

[4.47213595499958,
 4.69041575982343,
 4.898979485566356,
 5.0990195135927845,
 5.291502622129181,
 5.477225575051661]

In [28]:
lista = pares.flatMap(lambda x: (x, math.sqrt(x))).collect()

print(lista)

[20, 4.47213595499958, 22, 4.69041575982343, 24, 4.898979485566356, 26, 5.0990195135927845, 28, 5.291502622129181, 30, 5.477225575051661]


In [30]:
sqrt20 = sqrt.repartition(20)

sqrt20.getNumPartitions()

20

In [33]:
sqrt5 = sqrt20.coalesce(5)

sqrt5.getNumPartitions()

5

In [34]:
rdd = sc.textFile("/content/transacciones")

In [36]:
rdd.collect()

['(1001, 52.3)',
 '(1005, 20.8)',
 '(1001, 10.1)',
 '(1004, 52.7)',
 '(1005, 20.7)',
 '(1002, 85.3)',
 '(1004, 20.9)']

In [37]:
def proceso(s):
  return (s.replace("(", "").replace(")", "").split(", "))

In [38]:
rdd_llave_valor = rdd.map(proceso)

In [39]:
rdd_llave_valor.collect()

[['1001', '52.3'],
 ['1005', '20.8'],
 ['1001', '10.1'],
 ['1004', '52.7'],
 ['1005', '20.7'],
 ['1002', '85.3'],
 ['1004', '20.9']]

In [40]:
rdd_llave_valor.reduceByKey(lambda x, y: float(x) + float(y)).collect()

[('1002', '85.3'), ('1001', 62.4), ('1005', 41.5), ('1004', 73.6)]